### Evaluation with Hugging Face evaluation

In [ ]:
from datasets import load_dataset
from transformers import pipeline
import evaluate
import numpy as np

# Load a known CNN/DailyMail dataset (summarization)
dataset = load_dataset("cnn_dailymail", "3.0.0", split="test")

# Define a generator (model)
summarizer = pipeline("summarization", model="google/flan-t5-small", tokenizer="google/flan-t5-small",
    device_map="auto", max_new_tokens=128, truncation=True,
)

def generate_summary(text: str) -> str:
    out = summarizer(text, do_sample=False)[0]["summary_text"]
    return out.strip()

# Run generation
preds, refs = [], []
for row in dataset:
    article = row["article"]
    reference = row["highlights"]
    pred = generate_summary(article)
    preds.append(pred)
    refs.append(reference)

# Load & compute metrics. ROUGE (lexical overlap) and BERTScore (semantic similarity
rouge = evaluate.load("rouge")
rouge_scores = rouge.compute(predictions=preds, references=refs)

bertscore = evaluate.load("bertscore")
bs = bertscore.compute(predictions=preds, references=refs, model_type="microsoft/deberta-large-mnli")

# Convert list scores to means for a compact report
bs_report = {
    "bertscore_precision": float(np.mean(bs["precision"])),
    "bertscore_recall":    float(np.mean(bs["recall"])),
    "bertscore_f1":        float(np.mean(bs["f1"])),
}


### Evaluation with Azure Evaluation SDK

In [ ]:
# pip install azure-ai-evaluation

from azure.ai.evaluation import (
    evaluate,
    RelevanceEvaluator,            # quality
    GroundednessEvaluator,         # quality (RAG / factual consistency)
    ContentSafetyEvaluator         # composite safety: hate/sexual/violence/self-harm
)

# Evaluator configuration
model_config = {
    "azure_openai_endpoint": "<YOUR_AOAI_ENDPOINT>",
    "api_key": "<YOUR_AOAI_KEY>",
    "deployment_name": "gpt-4o-mini"
}

# Instantiate evaluators
relevance = RelevanceEvaluator(model_config)
groundedness = GroundednessEvaluator(model_config)   # for RAG answers with context
safety = ContentSafetyEvaluator(get_azure_ai_project())    # composite safety (hate, sexual, violence, self-harm)

# run the batch evaluation and optionally log to Azure AI project
result = evaluate(
    data="eval_data.jsonl",
    evaluators={
        "relevance": relevance,
        "groundedness": groundedness,
        "safety": safety
    },
    evaluator_config=get_evaluator_config(),
    azure_ai_project=azure_ai_project,   # logs a run you can inspect in Azure AI Foundry
    output_path="./eval_summary.json"    # saves aggregate + per-row metrics locally
)